# Prediction Histograms

Acest notebook genereaza histograme pentru fiecare regresie (`Linear Regression`, `Ridge`, `Lasso`, `XGBoost`), comparand valorile `Actual Final Price` cu valorile `Predicted Final Price`.

Sunt incluse doua seturi de grafice:
- fara eliminarea valorilor garbage
- cu eliminarea valorilor garbage


In [ ]:
%pip install matplotlib
import math
import matplotlib.pyplot as plt
import pandas as pd

from prediction_engine import MODEL_OPTIONS, build_prediction_table_from_dataframe, load_excel

DATA_FILE = "Project 2 Data.xlsx"
MONTHS_BACK = None  # pune un numar, de exemplu 6, daca vrei doar ultimele N luni
MODEL_ORDER = ["linear", "ridge", "lasso", "xgboost"]
MODEL_COLORS = {
    "linear": "#1f77b4",
    "ridge": "#2ca02c",
    "lasso": "#ff7f0e",
    "xgboost": "#d62728",
}
ACTUAL_COLOR = "#444444"

df = load_excel(DATA_FILE)
print(f"Rows loaded: {len(df)}")


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def build_results(ignore_garbage):
    results = {}

    for model_key in MODEL_ORDER:
        result_frame = build_prediction_table_from_dataframe(
            df=df,
            model_key=model_key,
            months_back=MONTHS_BACK,
            ignore_garbage=ignore_garbage,
        )
        results[model_key] = result_frame[result_frame["Status"] == "OK"].copy()

    return results


def plot_histograms(results_by_model, title):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for index, model_key in enumerate(MODEL_ORDER):
        ax = axes[index]
        result_frame = results_by_model[model_key]

        actual_values = result_frame["Actual Final Price"].dropna()
        predicted_values = result_frame["Predicted Final Price"].dropna()

        bins_count = 30
        if len(actual_values) > 1 or len(predicted_values) > 1:
            ax.hist(
                actual_values,
                bins=bins_count,
                alpha=0.55,
                color=ACTUAL_COLOR,
                label="Actual",
            )
            ax.hist(
                predicted_values,
                bins=bins_count,
                alpha=0.45,
                color=MODEL_COLORS[model_key],
                label="Predicted",
            )

        ax.set_title(MODEL_OPTIONS[model_key])
        ax.set_xlabel("Final Price")
        ax.set_ylabel("Number of Groups")
        ax.grid(axis="y", alpha=0.2)
        ax.legend()

    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


def build_summary_table(results_by_model, label):
    rows = []

    for model_key in MODEL_ORDER:
        result_frame = results_by_model[model_key]
        rows.append(
            {
                "Scenario": label,
                "Model": MODEL_OPTIONS[model_key],
                "Groups": len(result_frame),
                "Mean Actual": result_frame["Actual Final Price"].mean(),
                "Mean Predicted": result_frame["Predicted Final Price"].mean(),
                "Mean Abs Difference": result_frame["Abs Difference"].mean(),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
results_without_garbage_removal = build_results(ignore_garbage=False)
plot_histograms(
    results_without_garbage_removal,
    "Histograme actual vs predicted fara eliminarea valorilor garbage",
)
build_summary_table(
    results_without_garbage_removal,
    "Fara garbage removal",
)


In [ ]:
results_with_garbage_removal = build_results(ignore_garbage=True)
plot_histograms(
    results_with_garbage_removal,
    "Histograme actual vs predicted cu eliminarea valorilor garbage",
)
build_summary_table(
    results_with_garbage_removal,
    "Cu garbage removal",
)
